# 03 · Extract evidence and detect missing information

**Question:** Which facts can a deterministic extractor recover, and where should it admit uncertainty?

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
get_ipython().run_line_magic('matplotlib', 'inline')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
DATA = ROOT / 'data/synthetic/transfer_requests.csv'
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False, 'axes.spines.right': False})


In [2]:
from transfer_assistant.extraction import RuleBasedExtractor
from transfer_assistant.examples import EXAMPLES
from transfer_assistant.schema import missing_fields
extractor = RuleBasedExtractor()
note = EXAMPLES['Conflicting pressor documentation']
result = extractor.extract(note)
print(note)
display(pd.DataFrame([{'field':k,'value':str(v),'evidence':' | '.join(result.evidence.get(k, []))} for k,v in result.values.items()]))
display({'conflicts':result.conflicts,'missing':missing_fields(result)})

SYNTHETIC CASE. 64-year-old adult. Presenting problem: hypotension. Service: internal medicine. Currently on room air. No vasopressors. Currently on norepinephrine. Telemetry required. No specialty evaluation required. SBP 94 mmHg. HR 105 bpm. SpO2 96%. Isolation: none.


,field,value,evidence
0,patient_age,64,64-year-old adult
1,presenting_problem,hypotension,Presenting problem: hypotension
2,requested_service,internal medicine,Service: internal medicine
3,requested_level_of_care,None,
4,oxygen_support,room air,Currently on room air
5,vasopressor_use,None,No vasopressors | Currently on norepinephrine
6,isolation_requirements,none,Isolation: none
7,relevant_consultants,None,
8,referring_facility,None,
9,transfer_urgency,None,


{'conflicts': ['vasopressor_use'],
 'missing': {'critical': ['vasopressor_use'],
  'operational': ['requested_level_of_care',
   'relevant_consultants',
   'referring_facility',
   'transfer_urgency',
   'bed_availability']}}

## Observable truth, not hidden truth
Score against note-visible annotations. An omitted or contradictory field is expected to be unavailable. Value precision counts incorrect non-null extraction as a false positive; value recall also counts it as a false negative. Exact-match accuracy includes correctly unknown fields.

In [3]:
from transfer_assistant.evaluation import extraction_metrics
data = pd.read_csv(DATA)
validation = data[data.split.eq('validation')].reset_index(drop=True)
results = [extractor.extract(n) for n in validation.free_text_transfer_note]
fields, errors, missing_scores = extraction_metrics(validation, results)
display(fields)
display(missing_scores)
display(errors.head())

,field,n,exact_match_accuracy,precision,recall,f1,tp,fp,fn
0,patient_age,125,1.0,1.0,1.0,1.0,121,0,0
1,presenting_problem,125,1.0,1.0,1.0,1.0,115,0,0
2,requested_service,125,1.0,1.0,1.0,1.0,115,0,0
3,requested_level_of_care,125,1.0,1.0,1.0,1.0,113,0,0
4,oxygen_support,125,1.0,1.0,1.0,1.0,118,0,0
5,vasopressor_use,125,1.0,1.0,1.0,1.0,117,0,0
6,isolation_requirements,125,1.0,1.0,1.0,1.0,116,0,0
7,relevant_consultants,125,1.0,1.0,1.0,1.0,114,0,0
8,referring_facility,125,1.0,1.0,1.0,1.0,115,0,0
9,transfer_urgency,125,1.0,1.0,1.0,1.0,112,0,0


{'unit': 'routing field per test note',
 'positive': 'unavailable (absent or conflicting)',
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'accuracy': 1.0,
 'confusion_matrix': [[1292, 0], [0, 83]]}

,request_id,field,expected,actual,note


In [4]:
for text in ['No vasopressors.', 'Vasopressor use: unknown.', 'History of prior mechanical ventilation.', 'If deterioration occurs, consider norepinephrine.']:
    parsed = extractor.extract(text)
    print(text, '=>', {k:parsed.values[k] for k in ['oxygen_support','vasopressor_use']})

No vasopressors. => {'oxygen_support': None, 'vasopressor_use': False}
Vasopressor use: unknown. => {'oxygen_support': None, 'vasopressor_use': None}
History of prior mechanical ventilation. => {'oxygen_support': None, 'vasopressor_use': None}
If deterioration occurs, consider norepinephrine. => {'oxygen_support': None, 'vasopressor_use': None}


## Open vocabulary and LLM extension
The shared `Extractor` protocol permits a later LLM implementation, but it must return the same nullable schema and source evidence. No schema-valid output is automatically true. Preserve downstream missingness, conflict validation, routing policy, and the final coordinator decision. Broader abbreviations and complex temporality need independently authored tests and clinically adjudicated evaluation.

In [5]:
display(pd.read_csv(ROOT/'reports/challenge_results.csv'))

,case,note,field,expected,actual,passed
0,explicit negative,No vasopressors.,vasopressor_use,False,False,True
1,absent support,Age: 61.,vasopressor_use,NaN,NaN,True
2,absent support,Age: 61.,oxygen_support,NaN,NaN,True
3,negated ventilation,Not intubated. Currently on room air.,oxygen_support,room air,room air,True
4,historical ventilation,History of prior mechanical ventilation.,oxygen_support,NaN,NaN,True
5,conditional support,"If deterioration occurs, consider norepinephrine.",vasopressor_use,NaN,NaN,True
6,conflicting pressors,No vasopressors. Currently on norepinephrine.,vasopressor_use,NaN,NaN,True
7,HFNC abbreviation,Currently on HFNC.,oxygen_support,high-flow nasal cannula,high-flow nasal cannula,True
8,NIV abbreviation,On BiPAP.,oxygen_support,noninvasive ventilation,noninvasive ventilation,True
9,numeric boundaries,Age: 54. Service: cardiology. HR 95. Isolation...,requested_service,cardiology,cardiology,True


## Limits
The authored challenge suite is a development diagnostic, not a population sample. Supported generator prose can score perfectly while unrecognized real-world phrasing fails. Keep the actual failed cases visible; do not convert expectations to match the parser.